# Test run: Prize-Collecting Skill-VRP — ALNS

Interaktiver Testlauf des finalen Solvers (entspricht `skillvrp.py`, aber
Schritt für Schritt): Instanz laden → Greedy-Start → ALNS → Checker.

Instanz, Laufzeit und Seed unten in der Konfigurationszelle anpassen.

In [ ]:
from pathlib import Path
import subprocess
import sys

from instance_reader import read_instance
from initial_solution import greedy_initial_solution
from alns import (
    run_alns,
    SAParams,
    ALNSParams,
    PenaltyParams,
    RandomRemoval,
    RelatedRemoval,
    WorstDensityRemoval,
    WorstDetourRemoval,
    SkillScarcityRemoval,
    GreedyBestInsertionRepair,
    Regret2InsertionRepair,
    SequentialCheapestInsertionRepair,
)

In [ ]:
# --- Konfiguration -------------------------------------------------------
instance_path = Path("../data/dataset/skillvrp_n400_v10_s8_k2.0_10.txt")
checker_path = Path("../data/checker.py")

runtime_seconds = 30.0
seed = 1

solution_dir = Path("solutions")
solution_dir.mkdir(parents=True, exist_ok=True)
solution_path = solution_dir / "solution.txt"

In [ ]:
inst = read_instance(str(instance_path))
print(f"Instanz: {inst.name}")
print(f"Kunden: {inst.num_customers}, Fahrzeuge: {inst.num_vehicles}, Skills: {inst.num_skills}")

In [ ]:
start_solution = greedy_initial_solution(
    inst,
    strategy="multi_start",
    verbose=True,
)

print()
print("Initial objective:", start_solution.objective)

In [ ]:
# --- Parameter (identisch zu skillvrp.py) --------------------------------

# T0 skaliert mit der Groesse einzelner Move-Deltas (mittlerer Kundenprofit),
# Abkuehlung erfolgt in run_alns zeitbasiert ueber das Laufzeitbudget.
profits = [inst.profit[c] for c in inst.customers]
mean_profit = sum(profits) / max(1, len(profits))

sa = SAParams(
    initial_temperature=max(10.0, 1.5 * mean_profit),
    min_temperature=0.4,
    cooling_rate=0.9995,  # unbenutzt: run_alns kuehlt zeitbasiert
    reheat_factor=0.75,
)

penalties = PenaltyParams(
    time_window_penalty=20.0,
    shift_penalty=20.0,
    skill_penalty=10000.0,
)

params = ALNSParams(
    random_seed=seed,
    segment_length=100,
    no_accept_limit=500,
    reaction_factor=0.20,
    min_operator_weight=0.05,
    score_global_best=25.0,
    score_better_current=10.0,
    score_accepted=3.0,
    score_rejected=0.0,
    time_cost_alpha=0.5,
    time_scale_seconds=0.01,
    verbose=True,
    return_to_best_limit=2000,
    polish_interval_seconds=0.5,
)

In [ ]:
# --- Operator-Portfolio (identisch zu skillvrp.py) ------------------------

# Zerstoerungsgroesse ist Teil der Operator-Identitaet: jede geometrische
# Variante existiert als "small" (Polieren) und "large" (Restrukturieren),
# die adaptiven Gewichte lernen den richtigen Mix pro Instanz.
small = (0.03, 0.12)
large = (0.12, 0.28)

destroy_operators = []
for label, fraction, max_remove in (("small", small, 40), ("large", large, 70)):
    variants = [
        RandomRemoval(fraction=fraction, min_remove=1, max_remove=max_remove,
                      initial_weight=1.0),
        RelatedRemoval(fraction=fraction, min_remove=1, max_remove=max_remove,
                       bias=3.0, initial_weight=1.0),
        WorstDetourRemoval(fraction=fraction, min_remove=1, max_remove=max_remove,
                           bias=3.0, initial_weight=1.0),
    ]
    for op in variants:
        op.name += "_" + label
    destroy_operators.extend(variants)

destroy_operators.extend([
    WorstDensityRemoval(fraction=(0.05, 0.18), min_remove=1, max_remove=50,
                        noise=0.05, initial_weight=1.0),
    SkillScarcityRemoval(fraction=(0.05, 0.18), min_remove=1, max_remove=50,
                         noise=0.05, initial_weight=1.0),
])

# Rausch-Amplitude der Noised Insertion, skaliert mit der Distanzgroesse
max_distance = max(map(max, inst.distance))
noise_amp = 0.1 * max_distance

repair_operators = [
    GreedyBestInsertionRepair(extra_unserved_limit=100, max_insertions=None,
                              min_delta_score=0.0, initial_weight=1.0),
    Regret2InsertionRepair(extra_unserved_limit=100, max_insertions=None,
                           min_delta_score=0.0, initial_weight=1.0),
    SequentialCheapestInsertionRepair(extra_unserved_limit=100, order="profit",
                                      initial_weight=1.0),
    SequentialCheapestInsertionRepair(extra_unserved_limit=100, order="random",
                                      initial_weight=1.0),
    SequentialCheapestInsertionRepair(extra_unserved_limit=100, order="profit",
                                      initial_weight=1.0, noise_amp=noise_amp),
]

print(f"{len(destroy_operators)} Destroy-, {len(repair_operators)} Repair-Operatoren")

In [ ]:
result = run_alns(
    runtime=runtime_seconds,
    inst=inst,
    start_solution=start_solution,
    sa=sa,
    params=params,
    penalties=penalties,
    destroy_operators=destroy_operators,
    repair_operators=repair_operators,
)

print()
print("ALNS fertig")
print("Best objective:", result.best_evaluation.profit)
print("Feasible:", result.best_evaluation.feasible)
print("Bediente Kunden:", result.best_evaluation.served_customers, "/", inst.num_customers)
print("Iterationen:", result.iterations)
print("Restarts:", result.restarts)
print("Laufzeit: %.1fs" % result.runtime_seconds)

In [ ]:
# Adaptive Operator-Statistik: wie oft wurde welcher Operator gezogen?
print(f"{'Destroy-Operator':32s} {'Gewicht':>8} {'Nutzungen':>10} {'avg Zeit':>9}")
for row in result.destroy_summary:
    print(f"{row['name']:32s} {row['weight']:>8.2f} {row['uses']:>10} {row['avg_time']*1000:>7.1f}ms")

print()
print(f"{'Repair-Operator':32s} {'Gewicht':>8} {'Nutzungen':>10} {'avg Zeit':>9}")
for row in result.repair_summary:
    print(f"{row['name']:32s} {row['weight']:>8.2f} {row['uses']:>10} {row['avg_time']*1000:>7.1f}ms")

In [ ]:
result.best_solution.write_for_checker(str(solution_path))
print("Loesung geschrieben nach:", solution_path)

In [ ]:
checker_result = subprocess.run(
    [sys.executable, str(checker_path), str(instance_path), str(solution_path)],
    capture_output=True,
    text=True,
)

print("Checker:")
print(checker_result.stdout)
if checker_result.stderr:
    print("stderr:", checker_result.stderr)
print("Return code:", checker_result.returncode, "(0 = gueltig)")